In [1]:
import pandas as pd
import numpy as np
import os,sys
import glob
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd

In [2]:
# LOAD NUTS SHAPEFILE

nuts= gpd.read_file(('/mnt/g/Heidelberg_hiwi/nuts3/NUTS_RG_20M_2006_4326/NUTS_RG_20M_2006_4326.shp'))
countries=nuts.CNTR_CODE[nuts.LEVL_CODE==3].reset_index(drop=True)

In [3]:
countries

0       PL
1       DE
2       DE
3       UK
4       PL
        ..
1457    FR
1458    FR
1459    FR
1460    FR
1461    ME
Name: CNTR_CODE, Length: 1462, dtype: object

In [5]:
# WILD FIRE RADIATING POWER

years=list(np.arange(2016,2023)) # FILES FROM 2016 to 2023

wf_eu=pd.DataFrame()
ds1=pd.read_csv(r'/mnt/g/Heidelberg_hiwi/Tareq/wild_fire/wildfire_rp_2015.csv') # FILE FROM 2015
names=ds1.iloc[:,0].reset_index(drop=True)
ds1=ds1.iloc[:,1:]
wf_eu=pd.concat([wf_eu,ds1],axis=1)
del(ds1)

for year in years:

    ds1=pd.read_csv(r'/mnt/g/Heidelberg_hiwi/Tareq/wild_fire/wildfire_rp_'+str(year)+'.csv')
    ds1=ds1.iloc[:,1:]
    wf_eu=pd.concat([wf_eu,ds1],axis=1)
    del(ds1)

wf_eu=wf_eu.T.reset_index().rename(columns=names)
wf_eu=wf_eu.rename(columns={'index':'Date'})
wf_eu['Date']=pd.to_datetime(wf_eu['Date'])
countries=nuts.CNTR_CODE[nuts.LEVL_CODE==3].reset_index(drop=True)
wf_eu_w=wf_eu.resample('7D', on='Date').mean().reset_index(drop=False)
wf_eu_w=pd.melt(wf_eu_w, id_vars=['Date'],var_name='District',value_name='Wild_fire_RP') # CHANGE NAME OF VARIABLE HERE
countries=countries.repeat(len(pd.unique(wf_eu_w.Date)))
wf_eu_w['Country'] = countries.values
    
wf_eu_w=wf_eu_w[['Date', 'Country', 'District', 'Wild_fire_RP']]  # CHANGE NAME OF VARIABLE HERE


dd=pd.unique(wf_eu_w.Date)
wf_eu=pd.DataFrame()

for ii in dd:

    aa=wf_eu_w[wf_eu_w.Date==ii].sort_values(by='Country')
    wf_eu=pd.concat([wf_eu,aa],axis=0)

# wf_eu.to_csv('/mnt/g/Heidelberg_hiwi/Tareq/wild_fire/CAMS_nuts3_wfrp_2015_2022.csv',index=False) # SAVE FILE  

# del(wf_eu,wf_eu_w)
del(wf_eu_w)

In [6]:
wf_eu

,Date,Country,District,Wild_fire_RP
305140,2015-01-02,AT,Mittelburgenland,0.0
247456,2015-01-02,AT,Waldviertel,0.0
253308,2015-01-02,AT,Weinviertel,0.0
327712,2015-01-02,AT,UnterkÃ¤rnten,0.0
326040,2015-01-02,AT,OberkÃ¤rnten,0.0
...,...,...,...,...
81091,2022-12-30,UK,Monmouthshire and Newport,0.0
47233,2022-12-30,UK,Northamptonshire,0.0
436809,2022-12-30,UK,Lancashire CC,0.0
35529,2022-12-30,UK,Derby,0.0


In [3]:
tareq=pd.read_csv('/mnt/g/Heidelberg_hiwi/Tareq/tareq_20240403.csv') # LAST DATA MATRIX WITH ALL VARIABLES

In [4]:
tareq

,Date,Country,District,Tmean,Tmin_min,Tmin,Tmax,Tmax_max,Precipitation,Wind_speed,Wind_direction,Relative_humidity,Solar_radiation,Cloud_fraction,Air_quality,Wild_fire_PM2_5,Wild_fire_RP,NAO,SPI
0,2015-01-01,AT,Mittelburgenland,-0.157157,-8.116206,-3.761211,2.614157,6.577174,17.256704,2.667062,278.333091,82.230430,215.013085,74.565019,13.892285,0.0,0.0,197.215714,0.943139
1,2015-01-01,AT,Waldviertel,-0.736939,-9.412203,-3.415324,1.282278,4.871542,22.776036,3.047592,228.588819,85.393880,153.240692,85.788323,10.249391,0.0,0.0,197.215714,1.117989
2,2015-01-01,AT,Weinviertel,0.398948,-6.263482,-2.527339,2.845147,7.492108,11.648080,3.408183,221.762499,79.712598,172.422082,79.320286,13.094108,0.0,0.0,197.215714,0.628499
3,2015-01-01,AT,UnterkÃ¤rnten,-2.851160,-10.213027,-6.928416,1.260433,3.976111,5.721072,0.993443,305.106755,79.187487,258.680532,61.610321,15.891058,0.0,0.0,197.215714,-0.036364
4,2015-01-01,AT,OberkÃ¤rnten,-4.555904,-12.841329,-8.125497,-0.576072,2.793743,12.482570,0.992071,315.023082,73.236007,214.651709,55.655121,5.815526,0.0,0.0,197.215714,0.158535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605991,2022-12-29,UK,Monmouthshire and Newport,8.355503,5.219802,6.385935,10.057154,11.401315,19.509420,5.872395,218.471497,87.513636,47.862288,85.930701,4.700786,0.0,0.0,93.665000,0.280739
605992,2022-12-29,UK,Northamptonshire,8.509046,4.487068,5.937940,10.535281,12.099532,7.267937,6.654013,217.181115,85.602453,47.592218,77.984232,4.694525,0.0,0.0,93.665000,-0.514543
605993,2022-12-29,UK,Lancashire CC,6.838715,3.912296,4.770721,8.285462,8.883673,35.720418,5.678373,214.382393,87.408274,29.276889,85.924449,4.809907,0.0,0.0,93.665000,0.531608
605994,2022-12-29,UK,Derby,7.936224,3.950294,5.333796,9.946854,11.100022,10.815006,6.306049,216.617731,85.469963,43.311951,72.754655,4.920300,0.0,0.0,93.665000,-0.457924


In [5]:
matching_id = wf_eu.District.isin(tareq.District) 

In [9]:
new_df = wf_eu.loc[matching_id, :].reset_index(drop=True)

In [11]:
tareq['Wild_fire_RP']=new_df.Wild_fire_RP